In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import pandas as pd

PATIENT_CSV = "/content/drive/MyDrive/気合のCOVID19/00_共通・研究管理/02_データ定義・元データ/患者データファイル.csv"
CXR_CSV = "/content/drive/MyDrive/気合のCOVID19/00_共通・研究管理/03_患者選定/CXR_最終選定画像一覧.csv"

patients = pd.read_csv(PATIENT_CSV)
cxr = pd.read_csv(CXR_CSV)

print("患者データ:", patients.shape)
print("最終CXR:", cxr.shape)

print("\n患者データ患者数:", patients["to_patient_id"].nunique())
print("最終CXR患者数:", cxr["Subject ID"].nunique())

患者データ: (1384, 131)
最終CXR: (1277, 13)

患者データ患者数: 1384
最終CXR患者数: 1277


In [3]:
cohort = cxr[["Subject ID"]].merge(
    patients[["to_patient_id", "last.status"]],
    left_on="Subject ID",
    right_on="to_patient_id",
    how="inner"
)

cohort["true_label"] = cohort["last.status"].map({
    "discharged": 0,
    "deceased": 1
})

print("コホート患者数:", len(cohort))
print("\n=== 院内転帰 ===")
print(cohort["last.status"].value_counts())

print("\n=== true_label ===")
print(cohort["true_label"].value_counts())

print("\n欠損数:", cohort["true_label"].isna().sum())

コホート患者数: 1277

=== 院内転帰 ===
last.status
discharged    1108
deceased       169
Name: count, dtype: int64

=== true_label ===
true_label
0    1108
1     169
Name: count, dtype: int64

欠損数: 0


In [4]:
from sklearn.model_selection import train_test_split

# まず 80% Train / 20% Temporary
train_df, temp_df = train_test_split(
    cohort,
    test_size=0.20,
    random_state=42,
    stratify=cohort["true_label"]
)

# Temporaryを半分に分けて Validation 10% / Test 10%
val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=42,
    stratify=temp_df["true_label"]
)

train_df = train_df.copy()
val_df = val_df.copy()
test_df = test_df.copy()

train_df["split"] = "train"
val_df["split"] = "val"
test_df["split"] = "test"

split_df = pd.concat(
    [train_df, val_df, test_df],
    ignore_index=True
)

print("=== split人数 ===")
print(split_df["split"].value_counts())

print("\n=== split × outcome ===")
print(pd.crosstab(
    split_df["split"],
    split_df["true_label"],
    margins=True
))

print("\n=== 死亡率 ===")
print(
    split_df.groupby("split")["true_label"]
    .agg(["count", "sum", "mean"])
)

=== split人数 ===
split
train    1021
val       128
test      128
Name: count, dtype: int64

=== split × outcome ===
true_label     0    1   All
split                      
test         111   17   128
train        886  135  1021
val          111   17   128
All         1108  169  1277

=== 死亡率 ===
       count  sum      mean
split                      
test     128   17  0.132812
train   1021  135  0.132223
val      128   17  0.132812


In [5]:
OUT = "/content/drive/MyDrive/気合のCOVID19/00_共通・研究管理/03_患者選定/COVID19_固定患者split_1277.csv"

split_df[
    ["Subject ID", "to_patient_id", "last.status", "true_label", "split"]
].to_csv(
    OUT,
    index=False,
    encoding="utf-8-sig"
)

print("保存完了:")
print(OUT)

保存完了:
/content/drive/MyDrive/気合のCOVID19/00_共通・研究管理/03_患者選定/COVID19_固定患者split_1277.csv
